## Popcorn Hack: Diagnose the Recursion

- **Faulty-call trace:** with `{3, 7}` and target 7, call 1 has `low = 0`, `high = 1`, and `mid = 0`. Since 7 > 3, the faulty call repeats `low = 0`, `high = 1`, and `mid = 0`.
- **Why the range fails to shrink:** `mid` is 0 and `low` is 0, so passing `mid` as the new `low` changes nothing. It never reaches the 7, and Java eventually runs out of stack.
- **Repair:** `return binarySearch(values, target, mid + 1, high);`, because `mid` has already been ruled out.
- **Missing-target test:** target 5 inspects index 0, then index 1, then reaches bounds `1, 0` and returns `-1`.
- **Merge-sort stages for `[9, 2, 6, 1]`:** singletons `[9] [2] [6] [1]`, sorted pairs `[2, 9]` and `[1, 6]`, final `[1, 2, 6, 9]`. Merging `[2, 4]` and `[2, 3]` gives `[2, 2, 3, 4]`, keeping both copies of 2.
- **My rule for a valid base case:** it must be reachable, and every recursive call must make the problem strictly smaller. For binary search, that means the range shrinks until `low > high`.

## Exit Ticket

1. Middle indexes 3, 1, 2, returning index 2.
2. It returns `-1`, and no other middle element is inspected.
3. `[1, 2, 4, 5, 8, 9]`.
4. Splitting only creates smaller groups. Merging is what puts values from different halves in order.
5. The empty `String`, which is the base case.

## Prediction

Traced by hand before running, using `ids = {3, 7, 12, 18, 24, 31, 42}`.

**1. Target 12**

| Inspection | low | high | mid | Value |
|---|---|---|---|---|
| 1 | 0 | 6 | 3 | 18 |
| 2 | 0 | 2 | 1 | 7 |
| 3 | 2 | 2 | 2 | 12 |

It returns index **2**.

**2. Missing target 20**

| Call | low | high | mid | Value |
|---|---|---|---|---|
| 1 | 0 | 6 | 3 | 18 |
| 2 | 4 | 6 | 5 | 31 |
| 3 | 4 | 4 | 4 | 24 |
| 4 | 4 | 3 | None | Empty range |

That is 4 calls but only 3 inspections, because the last call returns **-1** before inspecting anything.

**3. Empty inputs.** An empty search starts with `low = 0` and `high = -1`, so `low > high` right away and it returns -1. An empty sort hits `values.length <= 1` and returns `[]`.

**4. `[9, 2, 6, 1]`**

```
split:  [9, 2] | [6, 1]
split:  [9] | [2]    [6] | [1]
merge:  [2, 9]
merge:  [1, 6]
merge:  [1, 2, 6, 9]
```

**5. `[2, 4, 2, 3]`.** The pairs are `[2, 4]` and `[2, 3]`. In the final merge, `left[i] <= right[j]` takes the left 2 first, then the right 2, then 3, then 4, giving `[2, 2, 3, 4]`. Both copies are kept, because merging only moves values and never drops them.

**6. My tests.** A singleton `{5}` searching for 5 returns index 0. For duplicates, `{2, 2, 2, 2, 2}` searching for 2 returns index 2, not 0, because binary search stops at the first `mid` that matches, and that doesn't have to be the first matching index.

## Solution

I used the supplied methods unchanged and replaced only the six predictions.

In [1]:
import java.util.Arrays;

public class TicketHomework {
    public static int binarySearch(int[] ids, int target, int low, int high) {
        System.out.println("call low=" + low + ", high=" + high);
        if (low > high) { return -1; }
        int mid = low + (high - low) / 2;
        System.out.println("inspect index " + mid + ", value " + ids[mid]);
        if (ids[mid] == target) { return mid; }
        if (target < ids[mid]) { return binarySearch(ids, target, low, mid - 1); }
        return binarySearch(ids, target, mid + 1, high);
    }

    public static int[] merge(int[] left, int[] right) {
        int[] result = new int[left.length + right.length];
        int i = 0, j = 0, k = 0;
        while (i < left.length && j < right.length) {
            if (left[i] <= right[j]) { result[k++] = left[i++]; }
            else { result[k++] = right[j++]; }
        }
        while (i < left.length) { result[k++] = left[i++]; }
        while (j < right.length) { result[k++] = right[j++]; }
        return result;
    }

    public static int[] mergeSort(int[] values) {
        if (values.length <= 1) { return values; }
        int mid = values.length / 2;
        int[] left = Arrays.copyOfRange(values, 0, mid);
        int[] right = Arrays.copyOfRange(values, mid, values.length);
        System.out.println("split " + Arrays.toString(values) + " -> "
            + Arrays.toString(left) + " | " + Arrays.toString(right));
        int[] result = merge(mergeSort(left), mergeSort(right));
        System.out.println("merge -> " + Arrays.toString(result));
        return result;
    }

    public static void check(String label, String prediction, String actual) {
        System.out.println((prediction.equals(actual) ? "PASS" : "FAIL")
            + " " + label + ": predicted " + prediction + ", actual " + actual);
    }

    public static void main(String[] args) {
        int predictedFound = 2;
        int predictedMissing = -1;
        int predictedEmpty = -1;
        String predictedSorted = "[1, 2, 6, 9]";
        String predictedDuplicates = "[2, 2, 3, 4]";
        String predictedEmptySort = "[]";

        int[] ids = {3, 7, 12, 18, 24, 31, 42};
        check("find 12", "" + predictedFound, "" + binarySearch(ids, 12, 0, ids.length - 1));
        check("find 20", "" + predictedMissing, "" + binarySearch(ids, 20, 0, ids.length - 1));
        check("empty search", "" + predictedEmpty, "" + binarySearch(new int[] {}, 12, 0, -1));
        check("sort", predictedSorted, Arrays.toString(mergeSort(new int[] {9, 2, 6, 1})));
        check("duplicates", predictedDuplicates, Arrays.toString(mergeSort(new int[] {2, 4, 2, 3})));
        check("empty sort", predictedEmptySort, Arrays.toString(mergeSort(new int[] {})));

        // My own tests from section 6.
        check("singleton", "0", "" + binarySearch(new int[] {5}, 5, 0, 0));
        check("duplicate target", "2", "" + binarySearch(new int[] {2, 2, 2, 2, 2}, 2, 0, 4));
    }
}

TicketHomework.main(null);

call low=0, high=6
inspect index 3, value 18
call low=0, high=2
inspect index 1, value 7
call low=2, high=2
inspect index 2, value 12
PASS find 12: predicted 2, actual 2
call low=0, high=6
inspect index 3, value 18
call low=4, high=6
inspect index 5, value 31
call low=4, high=4
inspect index 4, value 24
call low=4, high=3
PASS find 20: predicted -1, actual -1
call low=0, high=-1
PASS empty search: predicted -1, actual -1
split [9, 2, 6, 1] -> [9, 2] | [6, 1]
split [9, 2] -> [9] | [2]
merge -> [2, 9]
split [6, 1] -> [6] | [1]
merge -> [1, 6]
merge -> [1, 2, 6, 9]
PASS sort: predicted [1, 2, 6, 9], actual [1, 2, 6, 9]
split [2, 4, 2, 3] -> [2, 4] | [2, 3]
split [2, 4] -> [2] | [4]
merge -> [2, 4]
split [2, 3] -> [2] | [3]
merge -> [2, 3]
merge -> [2, 2, 3, 4]
PASS duplicates: predicted [2, 2, 3, 4], actual [2, 2, 3, 4]
PASS empty sort: predicted [], actual []
call low=0, high=0
inspect index 0, value 5
PASS singleton: predicted 0, actual 0
call low=0, high=4
inspect index 2, value 2
PASS

## Tests

The six supplied checks use the predictions above without changes, and the last two checks are my own.

## Feedback and Revision

TODO: get feedback from a peer, then record it and any revision here. Also rerun the published page, as the lesson asks.